# **Lab Report: Shooting method to solve Blasius equation**
<h3 align="RIGHT">Submitted by: Kalyana Raama Bheema<h3>

## Problem statement

For steady, incompressible flow of a uniform stream $U_\infty$ over a flat plate, the similarity variable $\eta = y\sqrt{U_\infty/(\nu x)}$ and $u/U_\infty = f'(\eta)$ turn the boundary layer equations into the **Blasius equation**:

$$f''' + \tfrac{1}{2} f f'' = 0, \qquad 0 < \eta < \infty$$

with boundary conditions

$$f(0) = 0, \qquad f'(0) = 0, \qquad f'(\eta) \to 1 \text{ as } \eta \to \infty$$

This is a **boundary value problem (BVP)**: two conditions are at $\eta = 0$ and one is at $\eta = \infty$. Runge–Kutta methods can only solve **initial value problems (IVPs)**, where all conditions are given at the start.

The **shooting method** fixes this: we guess the missing starting value, integrate forward, check how far we miss the condition at infinity, and correct the guess until we hit it.

In [5]:
# do not change line below
from google.colab import files

# import required packages here
import numpy as np
import matplotlib.pyplot as plt

## Step (a): Write the equation as a first-order system

The Blasius equation is third order, but Runge–Kutta works on first-order equations. So we define a vector

$$\mathbf{Y} = \begin{pmatrix} Y_1 \\ Y_2 \\ Y_3 \end{pmatrix} = \begin{pmatrix} f \\ f' \\ f'' \end{pmatrix}$$

and differentiate each component:

- $Y_1' = f' = Y_2$
- $Y_2' = f'' = Y_3$
- $Y_3' = f''' = -\tfrac{1}{2} f f'' = -\tfrac{1}{2} Y_1 Y_3$ (from the Blasius equation)

So the system is

$$\mathbf{Y}' = \mathbf{G}(\mathbf{Y}) = \begin{pmatrix} Y_2 \\ Y_3 \\ -\tfrac{1}{2} Y_1 Y_3 \end{pmatrix}$$


In [2]:
def G(Y):
    f, df, d2f = Y               # f, f', f''
    return np.array([df, d2f, -0.5 * f * d2f])

### Now, The RK3 and RK4 steps

**RK3**: Third order Runge Kutta With step size $h = \Delta\eta$:

$$k_1 = G(Y_n), \quad k_2 = G\left(Y_n + \tfrac{h}{2}k_1\right), \quad k_3 = G\left(Y_n - h k_1 + 2h k_2\right)$$
$$Y_{n+1} = Y_n + \tfrac{h}{6}\left(k_1 + 4k_2 + k_3\right)$$

Runge-Kutta's RK3 is third-order accurate: halving $h$ reduces the error by a factor of 8. It requires three evaluations of $f$ per step.

**RK4**: Fourth order Runge-Kutta Method

$$k_1 = G(Y_n), \quad k_2 = G\left(Y_n + \tfrac{h}{2}k_1\right), \quad k_3 = G\left(Y_n + \tfrac{h}{2}k_2\right), \quad k_4 = G(Y_n + h k_3)$$
$$Y_{n+1} = Y_n + \tfrac{h}{6}\left(k_1 + 2k_2 + 2k_3 + k_4\right)$$

Runge-Kutta's RK4 is fourth-order accurate: halving $h$ reduces the error by a factor of 16. It requires four evaluations of $f$ per step.

In [3]:
def rk3_step(Y, h):
    k1 = G(Y)
    k2 = G(Y + h/2 * k1)
    k3 = G(Y - h*k1 + 2*h*k2)
    return Y + h/6 * (k1 + 4*k2 + k3)

def rk4_step(Y, h):
    k1 = G(Y)
    k2 = G(Y + h/2 * k1)
    k3 = G(Y + h/2 * k2)
    k4 = G(Y + h * k3)
    return Y + h/6 * (k1 + 2*k2 + 2*k3 + k4)